# 02｜DETR 输入表示：Backbone、特征图与二维位置编码

上一课先从整体上认识了 DETR：它用固定数量的 Object Queries 直接预测一个物体集合。

现在正式进入 DETR 内部，并且一次只拆一段数据流。本课从输入图片开始，一直走到 Transformer Encoder 的入口：

```text
不同大小的图片
-> 补齐成一个 batch，并记录 padding mask
-> CNN Backbone 提取特征图
-> 1 x 1 卷积投影通道
-> 展平空间位置，得到图像 token 序列
-> 加入二维位置编码
-> 送入 Transformer Encoder
```

本课暂时不讲 Encoder 内部计算，也不讲 Object Queries 和 Decoder。先把 Encoder 到底接收什么弄清楚。

## 1. 为什么不能把原始图片直接交给 Transformer

一张 RGB 图片的形状是：

$$
B\times3\times H\times W
$$

其中每个空间位置只有 3 个颜色通道。单个像素的信息太原始：它可能表示物体边缘、背景纹理或光照变化，但仅凭一个像素很难判断这里是不是一只猫。

Transformer 擅长让一组已经具有一定含义的表示交换信息，却不负责从原始像素开始高效地提取局部视觉模式。

因此，原始 DETR 先使用成熟的 CNN Backbone：

- 浅层卷积提取边缘、颜色和纹理。
- 深层卷积组合出物体部件和更强的语义特征。
- 下采样缩小空间尺寸，避免 Transformer 处理过长的序列。

可以先记住：**CNN 负责把像素翻译成视觉特征，Transformer 负责让不同位置的视觉特征建立全局关系。**

## 2. 一个 batch 中的图片为什么需要 padding

目标检测数据集中的图片通常大小不同。例如：

- 图片 A：$480\times640$。
- 图片 B：$600\times800$。

PyTorch 中同一个 batch 通常要组成一个规则张量，因此需要把较小的图片补齐到本批次的最大高度和最大宽度。补齐后，两张图片都可以放入形状为：

$$
B\times3\times H_{max}\times W_{max}
$$

的张量中。

但补出来的区域不是真实图像内容。如果不告诉模型，Attention 也会把这些无效位置当成正常图像位置。

所以 DETR 会同时维护一个 **padding mask**：

- 有效图像区域：允许参与计算。
- padding 区域：在 Attention 中需要被忽略。

在原始实现常见的布尔 mask 约定中，`False` 表示有效位置，`True` 表示 padding 位置。阅读其他实现时要重新确认，因为不同代码库可能使用相反约定。

## 3. CNN Backbone 输出的不是检测框

原始 DETR 常以 ResNet-50 或 ResNet-101 作为 Backbone。Backbone 的任务只是提取特征，不会直接输出物体类别或边界框。

设输入图片为：

$$
x\in\mathbb{R}^{B\times3\times H\times W}
$$

经过 Backbone 后得到：

$$
f\in\mathbb{R}^{B\times C\times H'\times W'}
$$

四个维度分别表示：

- $B$：batch 大小。
- $C$：特征通道数。
- $H'$：特征图高度。
- $W'$：特征图宽度。

特征图上的一个位置不再对应一个像素，而是对应原图中的一片区域。该位置的 $C$ 维向量描述这片区域提取到的视觉特征。

## 4. 下采样为什么既有好处也有代价

以常见的 stride 32 为例，Backbone 输出特征图的高和宽大约是输入图片的 $1/32$。

假设输入经过整理后是 $640\times960$：

$$
H'=640/32=20,\qquad W'=960/32=30
$$

于是特征图只有 $20\times30=600$ 个空间位置。

### 好处

如果直接把原图每个像素都当成 token，会有 $640\times960=614400$ 个 tokens，全局 Attention 的关系矩阵大得无法接受。下采样后只剩 600 个 tokens，计算量大幅下降。

### 代价

一个较深特征位置覆盖原图中的较大区域，精细空间信息会减少。特别小的物体可能只占一个位置甚至不足一个位置。

这正是原始 DETR 对小目标相对不够理想的原因之一。现在先理解这个矛盾，不提前展开改进模型。

## 5. 为什么需要 1 x 1 卷积做通道投影

ResNet-50 最后阶段的特征图通常有 2048 个通道，而原始 DETR 的 Transformer 隐藏维度常设为 256。

两者维度不同，不能直接送入 Transformer。因此 DETR 使用一个 $1\times1$ 卷积：

$$
B\times2048\times H'\times W'
\rightarrow
B\times256\times H'\times W'
$$

$1\times1$ 卷积在这里主要完成通道维度的线性投影：

- 空间高度 $H'$ 不变。
- 空间宽度 $W'$ 不变。
- 每个位置的 2048 维特征被映射为 256 维。

它很像对特征图的每个空间位置共享同一个 `Linear(2048, 256)`，只是保持了 CNN 常用的张量布局。

这里的 256 记作 $D$，也就是 Transformer 的 `d_model`。

## 6. 从二维特征图变成图像 token 序列

Transformer 接收的是一组位置表示，而 CNN 输出是二维网格特征图，所以需要把空间维度展平。

投影后的特征图形状为：

$$
B\times D\times H'\times W'
$$

先把 $H'$ 和 $W'$ 合并成序列长度 $L$：

$$
L=H'W'
$$

再调整维度顺序：

$$
B\times D\times H'\times W'
\rightarrow B\times D\times L
\rightarrow B\times L\times D
$$

现在，序列中的每个 token 都来自特征图上的一个空间位置。注意：这里不是像 ViT 那样直接切原图 patch，而是把 **CNN 已经提取过的特征图位置**当成 tokens。

## 7. 用具体数字完整走一次 shape

假设：

- batch 大小 $B=2$。
- 补齐后的输入大小是 $640\times960$。
- Backbone 最终通道数 $C=2048$。
- 总下采样倍数约为 32。
- Transformer 隐藏维度 $D=256$。

形状变化为：

| 步骤 | 形状 | 解释 |
|---|---|---|
| 输入 batch | $2\times3\times640\times960$ | 两张补齐后的 RGB 图片 |
| Backbone 输出 | $2\times2048\times20\times30$ | 深层 CNN 特征图 |
| 1 x 1 投影 | $2\times256\times20\times30$ | 通道对齐到 $D=256$ |
| 合并空间维度 | $2\times256\times600$ | $20\times30=600$ |
| 调整维度顺序 | $2\times600\times256$ | 600 个图像 tokens |

因此，Encoder 看到的不是 640 x 960 个原始像素，而是 600 个经过 CNN 编码的视觉位置，每个位置由 256 维向量表示。

## 8. 展平之后，空间位置发生了什么

假设特征图是 $2\times3$ 的网格：

```text
(a00) (a01) (a02)
(a10) (a11) (a12)
```

展平后可能变成：

```text
a00, a01, a02, a10, a11, a12
```

数值向量被保留下来了，但序列本身只是一串位置。仅仅看到 `a04` 这样的序号，并不能让 Attention 天然理解：

- 它原来位于第几行、第几列。
- 哪些 token 在它的上、下、左、右。
- 两个位置在图像中相距多远。

展平只是改变张量排列方式，不会自动把二维坐标写入特征向量。这就是位置编码出现的原因。

## 9. 为什么 Self-Attention 天然不知道位置

Self-Attention 主要根据 token 内容生成 Q、K、V，并用内容相似度决定信息交换。

如果不提供位置编码，把输入 tokens 的顺序整体打乱，Attention 会跟着同样的排列处理它们，但不会意识到“这个特征从左上角移动到了右下角”。

这种性质可以理解为：Self-Attention 能处理一组元素之间的内容关系，但不知道这些元素原来位于二维图像的什么地方。

目标检测恰恰高度依赖位置：

- 边界框要预测物体在哪里。
- 同样的纹理出现在左上角和右下角，空间含义不同。
- 物体部件之间具有上下、左右和远近关系。

因此，DETR 必须向每个图像 token 注入二维位置信息。

## 10. 一维位置编码为什么还不够直观

在文本 Transformer 中，token 原本沿一条线排列，只需要一个位置编号：第 0 个词、第 1 个词、第 2 个词。

图像位置却有两个坐标：

$$
(y,x)=(\text{第几行},\text{第几列})
$$

同一个展平序号依赖特征图宽度才能还原成行列坐标。例如序号 7：

- 在宽度为 4 的网格中，它位于第 1 行、第 3 列。
- 在宽度为 5 的网格中，它位于第 1 行、第 2 列。

所以 DETR 使用二维位置编码，分别表达纵向位置 $y$ 和横向位置 $x$，再组合成每个位置的完整编码。

## 11. DETR 的二维正弦余弦位置编码

原始 DETR 常用二维正弦余弦位置编码。它和学习 Transformer 时见过的正弦余弦位置编码思想相同，只是要同时编码行坐标与列坐标。

可以把构造过程分成四步：

1. 为每个有效位置确定纵向坐标 $y$。
2. 为每个有效位置确定横向坐标 $x$。
3. 使用多组不同频率的正弦和余弦函数分别编码 $y$ 与 $x$。
4. 拼接两部分，得到 $D$ 维位置向量。

因此位置编码张量与投影后的特征图拥有兼容形状：

$$
pos\in\mathbb{R}^{B\times D\times H'\times W'}
$$

展平后则是：

$$
pos\in\mathbb{R}^{B\times L\times D}
$$

位置编码不会增加 token 数量，也不会改变每个 token 的最终维度。它只是为每个位置提供一个可区分的空间身份。

## 12. 为什么要使用多种频率

如果只把一个坐标数字重复 256 次，表达方式非常单一。正弦余弦位置编码让不同通道使用不同变化速度：

- 高频通道随位置变化较快，善于区分相邻位置。
- 低频通道随位置变化较慢，能表达较大范围的位置变化。

把多种频率组合起来后，每个二维位置会得到一组独特而连续的空间描述。

这里不需要先背完整公式。当前真正重要的是理解：

> 内容特征回答“这里看到了什么”，位置编码回答“这些特征来自哪里”。

只有两者结合，Attention 才能同时利用语义与空间关系。

## 13. padding mask 也会影响位置编码

回到一个 batch 中大小不同的图片。较小图片的右侧或下方可能存在补齐区域，这些区域不能被当成真实坐标继续编码。

原始 DETR 的位置编码实现会利用 mask，在有效区域上沿行和列累计坐标，并把 padding 区域排除在正常图像内容之外。

随后，同一份 mask 还会被缩放到 Backbone 特征图大小，并展平成：

$$
B\times L
$$

传给 Transformer 作为 `key_padding_mask`。这样 Encoder 在计算 Attention 时就能忽略由 padding 产生的无效 tokens。

所以 mask 不是附属信息。它同时帮助模型正确处理：

- 不同大小图片组成 batch。
- 有效二维坐标的构造。
- Attention 对无效位置的屏蔽。

## 14. 位置编码究竟怎样进入 Attention

概念上常说“把内容特征和位置编码相加”，这样最容易理解。原始 DETR 的实现会把内容特征 `src` 和位置编码 `pos` 分开保存，并在 Encoder Self-Attention 计算 Q、K 时加入位置：

$$
Q=(src+pos)W_Q
$$

$$
K=(src+pos)W_K
$$

而 V 主要携带内容信息：

$$
V=srcW_V
$$

这样，Q 与 K 计算相关性时既能考虑“内容像不像”，也能考虑“位置在哪里”；真正被加权汇总的 V 仍以视觉内容为主。

阅读不同 DETR 实现时，具体写法可能略有差异，但核心目标不变：让 Attention 的关系计算能够感知二维空间位置。

## 15. Encoder 入口处到底有哪几样东西

走到 Encoder 门口时，至少要区分三样信息：

| 名称 | 典型概念形状 | 作用 |
|---|---|---|
| `src` | $B\times L\times D$ | CNN 提取并投影后的图像内容 |
| `pos` | $B\times L\times D$ | 每个 token 的二维空间身份 |
| `mask` | $B\times L$ | 标记哪些 token 来自 padding |

较早的 PyTorch Transformer 接口常把序列维放在前面，因此原始代码内部也可能写成：

$$
L\times B\times D
$$

这与 $B\times L\times D$ 表达的是同一批数据，只是维度顺序不同。看到两种写法时不要误以为模型结构改变了。

Object Queries 也会进入 Transformer，但它们属于 Decoder 一侧，下一阶段再单独讲。

## 16. 与 ViT 和 Swin 的输入方式对比

| 对比角度 | ViT | 原始 DETR | Swin Transformer |
|---|---|---|---|
| token 来源 | 原图 patches 的线性投影 | CNN 特征图的空间位置 | 局部 patch tokens |
| 主要任务 | 图像分类 | 目标检测 | 分类或下游视觉任务 |
| 位置处理 | 常用可学习位置编码 | 常用二维正弦余弦编码 | 窗口内相对位置偏置 |
| 空间层级 | 通常单一尺度 | 原始版本主要使用深层特征 | 多 Stage 层级特征 |
| 特殊输出 token | 常见 CLS token | Decoder 使用 Object Queries | 分类时常用全局池化 |

最容易混淆的是：**Object Query 不是图像 token。**

- 图像 tokens 来自输入图片，数量由特征图大小决定。
- Object Queries 是可学习参数，数量由模型配置决定。
- 图像 tokens 进入 Encoder；Object Queries 主要进入 Decoder。

## 17. 常见误区

### 误区一：Backbone 已经完成了目标检测

Backbone 只输出特征图。最终类别和边界框由 Decoder 表示经过预测头产生。

### 误区二：1 x 1 卷积在切 patch

这里的 $1\times1$ 卷积不改变空间网格，只负责把通道数 $C$ 投影到 $D$。

### 误区三：展平后模型自动知道二维位置

展平只改变张量布局，不会自动提供行列坐标，所以还需要二维位置编码。

### 误区四：mask 是在遮住不想检测的背景

这里的 padding mask 只标记为了组成 batch 而补出来的无效区域。真实图片中的背景依然是有效输入。

### 误区五：图像 tokens 和 Object Queries 数量相同

两者互不绑定。图像 token 数是 $H'W'$，Query 数是人为设定的固定上限槽位数。

## 18. 本节小结

这一课需要真正记住七个结论：

1. 原始 DETR 先用 CNN Backbone 把原始像素变成深层视觉特征。
2. Backbone 下采样显著缩短 token 序列，但也会损失小目标所需的精细空间信息。
3. $1\times1$ 卷积只投影通道，将 Backbone 的 $C$ 对齐到 Transformer 的 $D$。
4. 特征图的 $H'\times W'$ 个位置展平后成为长度 $L=H'W'$ 的图像 token 序列。
5. Self-Attention 只根据内容无法知道 token 原来的二维位置，因此需要二维位置编码。
6. padding mask 负责标记补齐区域，并在 Attention 中屏蔽无效 tokens。
7. Encoder 入口要区分图像内容 `src`、二维位置 `pos` 和无效区域 `mask`。

完整 shape 主线是：

$$
B\times3\times H\times W
\rightarrow B\times C\times H'\times W'
\rightarrow B\times D\times H'\times W'
\rightarrow B\times L\times D
$$

其中 $L=H'W'$。下一课再进入 Transformer Encoder，逐步观察 Self-Attention 如何让图像中相隔很远的位置直接交换信息。

## 19. 自测问题

1. 为什么 DETR 不直接把 RGB 像素序列送进 Transformer？
2. 一个 batch 中的图片大小不同时，padding 和 mask 分别负责什么？
3. Backbone 输出的四个维度 $B\times C\times H'\times W'$ 各表示什么？
4. Backbone 下采样对 Transformer 计算量有什么帮助？
5. 下采样为什么可能伤害小目标检测？
6. $1\times1$ 卷积改变哪些维度，又保持哪些维度？
7. 特征图大小为 $20\times30$ 时，展平后有多少个图像 tokens？
8. 为什么展平操作本身不能代替位置编码？
9. DETR 的二维位置编码要分别表示哪两个坐标？
10. 内容特征和位置编码分别回答什么问题？
11. padding mask 展平后的形状是什么？
12. $B\times L\times D$ 和 $L\times B\times D$ 一定表示不同模型吗？
13. 图像 tokens 与 Object Queries 的来源有什么不同？
14. Encoder 入口的 `src`、`pos`、`mask` 分别有什么作用？

### 自测参考答案

1. 单个 RGB 像素语义太弱，而且像素序列过长；CNN 更适合先提取局部视觉特征并下采样。
2. padding 把图片补到统一大小；mask 标出补出来的无效区域。
3. 分别是 batch、特征通道、特征图高度和特征图宽度。
4. 它减少空间位置数量，从而缩短 Attention 的序列长度。
5. 小物体在深层低分辨率特征图上可能只占极少位置，细节容易丢失。
6. 它把通道数 $C$ 改为 $D$，保持 batch、高度和宽度不变。
7. $20\times30=600$ 个。
8. 展平只重新排列数据，不会把原来的行列坐标写进 token 表示。
9. 纵向行坐标 $y$ 和横向列坐标 $x$。
10. 内容特征回答“看到了什么”，位置编码回答“来自哪里”。
11. $B\times L$。
12. 不一定，通常只是 batch 维与序列维的排列顺序不同。
13. 图像 tokens 来自输入图像的 CNN 特征图；Object Queries 是模型学习的固定数量参数。
14. `src` 提供视觉内容，`pos` 提供二维位置，`mask` 屏蔽 padding tokens。